# PASSAGE v1.0 across its seven resolutions

This notebook gives one view of the whole release. It counts the scenes of every split and resolution
from the Parquet files, reports the outcome of the reference routes from the release receipt, and
shows how route length and search effort grow with the grid. The last part reads the metadata of the
first validation scenes of each resolution.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads Parquet footers and a few thousand metadata records; about one minute.

In [ ]:
resolutions = [64, 128, 256, 512, 1024, 2048, 4096]
split = "validation"
metadata_sample = 256  # validation scenes per resolution read in Section 3
output_dir = "notebook-artifacts/notebooks/paper/dataset_statistics"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "65f1e4f0a37d8a7b476ddf02d48fe542542924ea"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))




def release_receipt(name):
    """Read a published release receipt from the local copy, or from the pinned Hub revision."""
    if source.root is not None:
        return json.loads((Path(source.root) / "release-receipts" / name).read_text())
    from huggingface_hub import hf_hub_download
    return json.loads(Path(hf_hub_download(dataset_repo, f"release-receipts/{name}", repo_type="dataset",
                                           revision=dataset_revision)).read_text())

## 1. Scenes per resolution and split

The counts come from the footers of the core Parquet files, so every row is counted without reading
the data. Preview rows of the `_samples` configurations are not included.

In [ ]:
from passage.notebook_reports import split_inventory_report

SPLITS = ("train", "calibration", "validation", "test")
inventory = pd.DataFrame(split_inventory_report(source, resolutions)["records"])
inventory["Resolution"] = [f"{r}×{r}" for r in inventory.resolution]
show(inventory[["Resolution", *SPLITS, "total"]].rename(columns={"total": "Total"}),
     "Table 1. Core scenes per resolution and split.")

fig, ax = plt.subplots(figsize=(8.6, 3.4))
fig.subplots_adjust(top=0.78)
left = np.zeros(len(inventory))
for part in SPLITS:
    ax.barh(inventory.Resolution, inventory[part], left=left, color=SPLIT_COLORS[part], height=0.6,
            label=part, edgecolor="white", linewidth=0.8)
    left += inventory[part].to_numpy()
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f"{v / 1e3:,.0f}k" if v else "0"))
for y, total in enumerate(inventory.total):
    ax.text(total + 8_000, y, f"{total:,}", va="center", fontsize=8, color=INK)
ax.invert_yaxis()
ax.set(xlabel="Core scenes", xlim=(0, inventory.total.max() * 1.18))
ax.grid(True, axis="x")
ax.grid(False, axis="y")
ax.legend(ncol=4, fontsize=8, loc="lower right")
figure_title(fig, f"{inventory.total.sum():,} core scenes in seven resolutions",
             "Split shares are 90 / 2 / 4 / 4 % at every resolution")
figures = save_figure(fig, output_path, "dataset_statistics-inventory")
plt.show()

*Figure 1. Core scenes per resolution, split by colour. The four smallest grids hold one million scenes
each; the larger ones hold fewer because each scene is much larger.*

## 2. Outcome of the reference routes

Every scene stores six route variants. The release receipt `publication-facts.json` records how many
are exact optima and how many are proved to have no path. Table 2 reproduces it.

In [ ]:
facts = release_receipt("publication-facts.json")["resolutions"]
outcomes = pd.DataFrame([{
    "Resolution": f"{r}×{r}", "Route variants": 6 * facts[str(r)]["requested_scenes"],
    "Exact routes": facts[str(r)]["successful_route_variants"],
    "Proved no-path": facts[str(r)]["proved_infeasible_variants"],
    "Unresolved": facts[str(r)]["unresolved_variants"]} for r in resolutions])
outcomes["No-path share (%)"] = 100 * outcomes["Proved no-path"] / outcomes["Route variants"]
show(outcomes, "Table 2. Outcome of the reference routes per resolution (release receipt).", "{:,.3f}")

fig, ax = plt.subplots(figsize=(8.6, 3.2))
fig.subplots_adjust(top=0.76)
ax.bar(outcomes.Resolution, outcomes["No-path share (%)"], color=REGIME_COLORS["obstacles"], width=0.55)
for x, value in enumerate(outcomes["No-path share (%)"]):
    ax.text(x, value + 0.004, f"{value:.3f} %", ha="center", fontsize=8, color=INK)
ax.set(ylabel="Proved no-path (% of variants)", ylim=(0, outcomes["No-path share (%)"].max() * 1.25))
figure_title(fig, "Proved no-path variants stay rare at every resolution",
             f"{outcomes['Proved no-path'].sum():,} of {outcomes['Route variants'].sum():,} variants")
figures += save_figure(fig, output_path, "dataset_statistics-no-path")
plt.show()

*Figure 2. Share of route variants proved to have no path. They occur only with obstacles, when the
obstacles separate the start from the goal.*

## 3. Route length and search effort

For each resolution, Section 3 reads the metadata of the first `metadata_sample` validation scenes. The
route length is the number of cells of the stored route. The search effort is the number of cells the
generation search marked as considered, as recorded in each row's metadata, divided by the cells of
the grid.

In [ ]:
SOLVERS = {"astar_elevation": "elevation", "astar_slope": "slope_uphill", "astar_energy": "energy"}
records = []
for resolution in resolutions:
    for row in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False, limit=metadata_sample):
        for solver, objective in SOLVERS.items():
            for condition in ("free", "obstacles"):
                path = row["metadata"]["paths"][solver][condition]
                if path["status"] == "ok":
                    records.append({"resolution": resolution, "objective": objective, "condition": condition,
                                    "length": path["length"], "considered": path["considered_area"]})
effort = pd.DataFrame(records)
effort["considered_share"] = 100 * effort.considered / effort.resolution ** 2
table = (effort.groupby("resolution")
         .agg(**{"Routes": ("length", "size"), "Median length (cells)": ("length", "median"),
                 "Median considered cells": ("considered", "median"),
                 "Median share of grid (%)": ("considered_share", "median")}).reset_index())
table.insert(0, "Resolution", [f"{r}×{r}" for r in table.pop("resolution")])
show(table, f"Table 3. Route length and recorded search effort, {metadata_sample} validation scenes per resolution.", "{:,.1f}")

fig, (ax_len, ax_share) = plt.subplots(1, 2, figsize=(11, 3.8))
fig.subplots_adjust(top=0.78, wspace=0.25)
for objective in SOLVERS.values():
    med = effort[effort.objective == objective].groupby("resolution").length.median()
    ax_len.plot(med.index, med.values, marker="o", markersize=5, color=OBJECTIVE_COLORS[objective],
                label=OBJECTIVE_LABELS[objective])
    share = effort[effort.objective == objective].groupby("resolution").considered_share.median()
    ax_share.plot(share.index, share.values, marker="o", markersize=5, color=OBJECTIVE_COLORS[objective],
                  label=OBJECTIVE_LABELS[objective])
for ax in (ax_len, ax_share):
    ax.set_xscale("log", base=2)
    ax.set_xticks(resolutions, [str(r) for r in resolutions])
    ax.set_xlabel("Grid side (cells)")
ax_len.set_yscale("log")
ax_len.set(ylabel="Median route length (cells)", title="Route length")
ax_share.set(ylabel="Median share of grid considered (%)", title="Search effort", ylim=(0, 100))
ax_len.legend(fontsize=8)
figure_title(fig, "Route length and recorded search effort across resolutions",
             f"{metadata_sample} {split} scenes per resolution · stored routes and recorded search effort")
figures += save_figure(fig, output_path, "dataset_statistics-scaling")
plt.show()

*Figure 3. Left: median route length against the side of the grid, both on log scales. Right: median
share of the grid that the generation search considered.*

## Summary

In [ ]:
slope = np.polyfit(np.log(table.index.map(lambda i: resolutions[i])), np.log(table["Median length (cells)"]), 1)[0]
display(Markdown("\n".join([
    f"- The release holds **{inventory.total.sum():,} core scenes** in {len(resolutions)} resolutions, with the same "
    "90 / 2 / 4 / 4 % split shares at each.",
    f"- {outcomes['Exact routes'].sum():,} route variants are exact optima and {outcomes['Proved no-path'].sum():,} "
    f"({100 * outcomes['Proved no-path'].sum() / outcomes['Route variants'].sum():.3f} %) are proved no-path; "
    f"{outcomes['Unresolved'].sum()} are unresolved.",
    f"- Median route length grows roughly as the grid side to the power {slope:.2f}, while the number of "
    "cells grows as its square: routes cover a shrinking share of the grid as the resolution rises.",
])))
report = {"inventory": inventory.drop(columns="Resolution").to_dict("records"),
          "route_outcomes": outcomes.to_dict("records"), "scaling": table.to_dict("records"),
          "length_exponent": float(slope), "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, "dataset_statistics.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*